# J1. Jev로 라벨 오류 후보 판정 (CPU 런타임)

**무엇을 하나**: 여러 모델이 한목소리로 정답 라벨과 다르게 답한 곳(`reports/label_error_candidates/test.jsonl`, src/find_label_errors.py 결과)을 TypeSafe의 판단 전용 모델 **Jev**에게 보여 주고 "이 구간은 7개 태그 중 무엇인가, 아니면 개체가 아닌가"를 고르게 한다. Jev는 글을 쓰지 않고 **선택지 + 확률 + 신뢰도**만 돌려준다.

**판정 규칙** (후보 종류별)
| 종류 | 정답 라벨 | 모델 답 | Jev에게 묻는 것 | 판정 |
|---|---|---|---|---|
| 누락 | 없음 | 구간 X를 태그 T로 | X는 무엇인가(7태그 + 개체 아님) | T면 '라벨 누락', 개체 아님이면 '라벨 맞음' |
| 과잉 | 구간 X를 태그 T로 | 없음 | X는 무엇인가 | 개체 아님이면 '라벨 오류(과잉)', T면 '라벨 맞음' |
| 유형 | X를 T1으로 | X를 T2로 | X는 무엇인가 | T2면 '라벨 태그 오류', T1이면 '라벨 맞음' |
| 경계 | 범위 A | 범위 B (같은 태그) | A와 B 중 개체의 정확한 범위는 | B면 '라벨 범위 오류', A면 '라벨 맞음' |

신뢰도가 낮으면(기본 0.5 미만) '애매'로 둔다. **Jev 판정은 정답이 아니다.** 사람이 검수한 골드셋으로 Jev 판정이 얼마나 맞는지 먼저 잰 뒤에 믿는다.

**순서**: 먼저 `N = 100`(종류별로 고르게)으로 돌려 사용량을 보고, 늘릴지 정한다. 비용은 TypeSafe 계정에서 나간다.


## 0. 준비
- 런타임: **CPU** (GPU 필요 없음).
- 키: Colab 왼쪽 열쇠 아이콘(보안 비밀)에 `TYPESAFE_API_KEY` 이름으로 넣고 '노트북 액세스'를 켠다. 노트북·파일·채팅에 키를 쓰지 않는다.


In [1]:
!pip -q install -U typesafe-sdk
import os, json, random, collections, time
from google.colab import drive, userdata
drive.mount("/content/drive")
os.environ["TYPESAFE_API_KEY"] = userdata.get("TYPESAFE_API_KEY")
from typesafe_sdk import Choice, TypeSafeClient
PROJECT = "/content/drive/Othercomputers/내 PC/NLP/프로젝트1/공통2_개체명인식_NER"
SPLIT   = "test"                      # 골드셋을 test에서 만들 예정이라 test 후보부터
N       = 100                         # 먼저 100개. 종류별로 고르게 뽑음
CONF_MIN = 0.5                        # 이보다 낮으면 '애매'
OUT_DIR = f"{PROJECT}/reports/jev"; os.makedirs(OUT_DIR, exist_ok=True)
OUT = f"{OUT_DIR}/jev_{SPLIT}_judgments.jsonl"
cands = [json.loads(l) for l in open(f"{PROJECT}/reports/label_error_candidates/{SPLIT}.jsonl", encoding="utf-8")]
print(len(cands), "후보", collections.Counter(c["kind"] for c in cands))


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 kB 1.0 MB/s eta 0:00:00
Mounted at /content/drive
2890 후보 Counter({'과잉': 1516, '누락': 836, '경계': 437, '유형': 101})


## 1. 질문 만들기
태그 설명은 AI Hub 공식 정의(reports/공식_태그정의_및_라벨링규칙.md)를 짧게 줄인 것이다. 이 데이터에서는 직업·직위도 CV로 라벨되어 있어 CV 설명에 넣었다.


In [2]:
TAG_CRITERIA = {
    "OG": "기관 및 조직. 기관·단체·회사·정당·팀, 회의/회담 명칭",
    "AF": "인공물. 사람이 만든 대상물: 작품·제품·건물·서비스 이름 등",
    "CV": "문명·문화 용어(직업·직위 포함): 제도·문화·직책 등을 가리키는 용어",
    "FD": "학문 분야. 과학·사회과학·의학·예술·철학 분야와 학파",
    "TR": "이론·법칙·원리·주의(主義)의 명칭",
    "TM": "위 정의 밖의 전문 용어(의학 용어·기술 용어 등)",
    "NW": "표준국어대사전과 우리말샘에 없는 신조어",
    "NONE": "개체가 아니다: 고유명이 아닌 일반 명사(단, 직업·직위는 CV), 동사·형용사의 일부, 잘못 자른 구간",
}
def mark(text, s, e): return text[:s] + "[[" + text[s:e] + "]]" + text[e:]
def find_span(text, s_txt, near=None):
    i = text.find(s_txt);  return (i, i + len(s_txt)) if i >= 0 else None

def build(c):
    t = c["text"]
    if c["kind"] in ("누락", "유형"):
        s, e, tag = c["model"]
        gold_tag = c["gold"][0].rsplit("/", 1)[1] if c["gold"] else None
        return {"span": (s, e), "state": f"문장: {mark(t, s, e)}\n검토할 구간: {t[s:e]}",
                "q": Choice(instructions="AI Hub 개체명 라벨링 지침(개체명은 원칙적으로 고유명, TTA 세분류에 있는 용어만)에 따르면, 문장에서 [[ ]]로 표시한 구간은 아래 중 무엇인가? 구간 전체가 하나의 개체일 때만 태그를 고른다.", criteria=TAG_CRITERIA),
                "model_tag": tag, "gold_tag": gold_tag}
    if c["kind"] == "과잉":
        s_txt, tag = c["gold"][0].rsplit("/", 1); sp = find_span(t, s_txt)
        if not sp: return None
        s, e = sp
        return {"span": (s, e), "state": f"문장: {mark(t, s, e)}\n검토할 구간: {t[s:e]}",
                "q": Choice(instructions="AI Hub 개체명 라벨링 지침(개체명은 원칙적으로 고유명, TTA 세분류에 있는 용어만)에 따르면, 문장에서 [[ ]]로 표시한 구간은 아래 중 무엇인가? 구간 전체가 하나의 개체일 때만 태그를 고른다.", criteria=TAG_CRITERIA),
                "model_tag": None, "gold_tag": tag}
    if c["kind"] == "경계":
        ms, me, tag = c["model"]; g_txt = c["gold"][0].rsplit("/", 1)[0]; sp = find_span(t, g_txt)
        if not sp: return None
        return {"span": (ms, me), "state": f"문장: {t}\n후보 A: {g_txt}\n후보 B: {t[ms:me]}\n태그: {tag} ({TAG_CRITERIA[tag]})",
                "q": Choice(instructions="이 문장에서 해당 태그 개체의 정확한 범위는 어느 쪽인가? (조사·어미는 빼고, 하나의 대상을 가리키는 말 전체)",
                            criteria={"A": f"'{g_txt}'", "B": f"'{t[ms:me]}'", "NEITHER": "둘 다 정확한 범위가 아니다"}),
                "model_tag": tag, "gold_tag": tag}

def verdict(c, ans):
    ch, conf = ans.choice, ans.confidence
    if conf is not None and conf < CONF_MIN: return "애매"
    k = c["kind"]
    if k == "누락":  return "라벨 누락(모델 맞음)" if ch == c["_b"]["model_tag"] else ("라벨 맞음" if ch == "NONE" else "다른 태그")
    if k == "과잉":  return "라벨 오류(개체 아님)" if ch == "NONE" else ("라벨 맞음" if ch == c["_b"]["gold_tag"] else "다른 태그")
    if k == "유형":  return "라벨 태그 오류(모델 맞음)" if ch == c["_b"]["model_tag"] else ("라벨 맞음" if ch == c["_b"]["gold_tag"] else "다른 태그")
    if k == "경계":  return "라벨 범위 오류(모델 맞음)" if ch == "B" else ("라벨 맞음" if ch == "A" else "둘 다 아님")

# 종류별로 고르게 N개 (고정 시드)
rng = random.Random(42); by = collections.defaultdict(list)
for c in cands: by[c["kind"]].append(c)
for v in by.values(): rng.shuffle(v)
sample, kinds = [], list(by)
while len(sample) < N and any(by.values()):
    for k in kinds:
        if by[k] and len(sample) < N: sample.append(by[k].pop())
print(collections.Counter(c["kind"] for c in sample))


Counter({'과잉': 25, '유형': 25, '누락': 25, '경계': 25})


## 2. Jev 호출
끊겨도 다시 실행하면 이미 판정한 후보는 건너뛴다(결과 파일에 한 줄씩 저장). 한 번 호출에 질문 하나(후보 하나)를 보낸다.


In [3]:
def key(c): return f"{c['id']}|{c['kind']}|{c['model']}|{c['gold']}"
done = {json.loads(l)["key"] for l in open(OUT, encoding="utf-8")} if os.path.exists(OUT) else set()
usage = collections.Counter(); n_err = 0; t0 = time.time()
with TypeSafeClient(model="jev-latest") as client, open(OUT, "a", encoding="utf-8") as f:
    for i, c in enumerate(sample):
        if key(c) in done: continue
        b = build(c)
        if b is None: continue
        c["_b"] = b
        try:
            r = client.system_one(state=b["state"], questions={"q": b["q"]})
        except Exception as ex:
            print("오류", type(ex).__name__, str(ex)[:200]); n_err += 1; continue
        a = r.answers["q"]
        u = getattr(r, "usage", None)
        if u is not None:
            usage["input_tokens"] += getattr(u, "input_tokens", 0) or 0; usage["output_tokens"] += getattr(u, "output_tokens", 0) or 0
        rec = {"key": key(c), "id": c["id"], "kind": c["kind"], "votes": c["votes"], "text": c["text"],
               "model": c["model"], "model_str": c["model_str"], "gold": c["gold"],
               "jev_choice": a.choice, "jev_confidence": a.confidence, "jev_probabilities": dict(a.probabilities),
               "verdict": verdict(c, a)}
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")
        if (i + 1) % 20 == 0: print(i + 1, "개", round(time.time() - t0), "초")
print("오류", n_err, "| 사용량", dict(usage), f"| {time.time()-t0:.0f}초")


20 개 3 초
40 개 5 초
60 개 8 초
80 개 11 초
100 개 15 초
오류 0 | 사용량 {'input_tokens': 75873, 'output_tokens': 6475} | 15초


## 3. 결과 요약
종류별 판정 분포와 신뢰도. '라벨 누락/오류' 비율이 높을수록 정답 라벨 쪽 문제가 많다는 뜻이다(단, Jev 정확도를 골드셋으로 재기 전에는 참고값).


In [4]:
import pandas as pd
R = [json.loads(l) for l in open(OUT, encoding="utf-8")]
df = pd.DataFrame(R)
display(pd.crosstab(df["kind"], df["verdict"], margins=True))
print("신뢰도 평균", round(df["jev_confidence"].mean(), 3), "| 0.5 미만", int((df["jev_confidence"] < CONF_MIN).sum()), "개")
for _, r in df.sample(min(8, len(df)), random_state=0).iterrows():
    print(f"\n[{r['kind']}] {r['verdict']} (Jev {r['jev_choice']}, 신뢰도 {r['jev_confidence']:.2f})")
    print("  문장:", r["text"][:120]); print("  모델:", r["model_str"], "| 정답:", r["gold"])
summary = {"split": SPLIT, "n": len(df), "by_kind_verdict": pd.crosstab(df["kind"], df["verdict"]).to_dict(),
           "mean_confidence": float(df["jev_confidence"].mean()), "usage": dict(usage)}
json.dump(summary, open(f"{OUT_DIR}/jev_{SPLIT}_summary.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
# 사람 검수용 시트 (골드셋 회의 뒤 Jev 정확도 측정에 씀)
df[["id", "kind", "text", "model_str", "gold", "jev_choice", "jev_confidence", "verdict"]].assign(사람판정="").to_csv(
    f"{OUT_DIR}/jev_{SPLIT}_review_sheet.csv", index=False, encoding="utf-8-sig")
print("저장:", OUT_DIR)


verdict,다른 태그,라벨 누락(모델 맞음),라벨 맞음,라벨 범위 오류(모델 맞음),라벨 오류(개체 아님),라벨 태그 오류(모델 맞음),애매,All
kind,,,,,,,,
경계,0,0,8,12,0,0,5,25
과잉,2,0,2,0,18,0,3,25
누락,2,8,13,0,0,0,2,25
유형,8,0,5,0,0,12,0,25
All,12,8,28,12,18,12,10,100


신뢰도 평균 0.784 | 0.5 미만 10 개

[누락] 라벨 누락(모델 맞음) (Jev OG, 신뢰도 0.92)
  문장: [인천=뉴시스] (이름) 기자 = sk인천석유화학은 지난 15일 인천시 서구청, 인천서부소방서와 공동으로 진행한 발달장애인을 위한 ‘hi, walk together’ 캠페인 달성 기부금 2000만원을 인천사회복지공동
  모델: 서구청 | 정답: []

[누락] 라벨 맞음 (Jev NONE, 신뢰도 0.95)
  문장: 대학기술지주회사란 연구 성과와 특허 기술을 사업화해 학생 창업을 돕든 영리법인이다.
  모델: 학생 | 정답: []

[누락] 라벨 맞음 (Jev NONE, 신뢰도 0.61)
  문장: 앞서 공급한 이들 단지들은 계약률이 각각 70% 후반대와 50% 수준인 것으로 알려졌다.분양 관계자는 '반도유보라와 비교할 때 3.3㎡당 20만원 정도 저렴하다'며 '특화설계를 도입하는 등 강점이 많아 정당계약 1개
  모델: 대표 | 정답: []

[경계] 라벨 범위 오류(모델 맞음) (Jev B, 신뢰도 0.74)
  문장: 개장 1년도 채 안돼…점포 64% 문닫아【진주=뉴시스】(이름) 기자 = 경남 진주시가 청년 먹거리 골목인 ‘청춘다락’을 개장한지 채 1년도 안돼 점포의 64%가 문을 닫아 활성화대책 마련이 시급하다.8일 진주시에 따
  모델: 진주시 | 정답: ['시/FD']

[경계] 라벨 범위 오류(모델 맞음) (Jev B, 신뢰도 0.89)
  문장: 신년 간담회, 해양 관광도시 조성·세일즈행정 등 약속 【목포=뉴시스】(이름) 기자 = 박홍률 전남 목포시장은 2일 국민의당 탈당에 이은 민주평화당 입당과 관련, '목포발전을 위한 결단'이라고 말했다.
  모델: 국민의당 | 정답: ['국민의당 탈당/OG']

[유형] 다른 태그 (Jev AF, 신뢰도 0.84)
  문장: 한편 (이름), (이름), (이름)이 출연하는 '코빅'은 오는 3일 2016년 1쿼터로 시작한다.
  모델: 코빅 | 정답: ['코빅/TM']

[과잉] 애매

## 4. 런타임 반납
결과 파일이 드라이브에 저장된 것을 확인하고 런타임을 반납한다.


In [ ]:
if os.path.exists(OUT) and os.path.getsize(OUT) > 0:
    drive.flush_and_unmount()
    from google.colab import runtime
    runtime.unassign()
else:
    print("결과 파일이 없어 반납하지 않습니다")
